In [11]:
import warnings
warnings.filterwarnings("ignore")
!pip install jupyterlab-lsp
!pip install "python-lsp-server[all]"
!pip install gradio
!pip install langchain langchain-core langchain-community langchain_groq


In [4]:
import requests
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/hTqGqoC-LrW6S79HjuJUkg/trimmed-02.wav"
response = requests.get(url)
audio_file_path = "sample-meeting.wav"
if response.status_code == 200:
    with open(audio_file_path,"wb") as file:
        file.write(response.content)
        print(f"File downloaded successfully")
else:
    print("Failed to download the file")

File downloaded successfully


In [6]:
import torch
from transformers import pipeline
pipe = pipeline(
    "automatic-speech-recognition",
    model = "openai/whisper-tiny.en",
    chunk_length_s = 30,
)
sample = "sample-meeting.wav"
prediction = pipe(sample, batch_size = 8)["text"]
print(prediction)


Device set to use cpu
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


 We have adopted a conservative approach to managing our leverage and have a healthy tier 1 capital ratio of 12.5%. Our forecast for the coming quarter is positive. We expect revenue to be around 135 million and 8% quarter over quarter growth driven primarily by our cutting its blockchain chain solutions and AI driven predictive analytics. We're also excited about the upcoming IPO of our FinTech subsidiary pay plus which we expect to raise 200 million significantly bolstering our liquidity and paving the way for aggressive growth strategies. We thank our shareholders for their continued faith in us and we look forward to an even more successful Q3. Thank you so much.


In [8]:
import gradio as gr

def transcript_audio(audio_file):
    pipe = pipeline(
        "automatic-speech-recognition",
        model = "openai/whisper-tiny.en",
        chunk_length_s = 30
    )
    result = pipe(audio_file , batch_size = 8)
    return result
audio_input = gr.Audio(sources = "upload",type = "filepath")
output_text = gr.Textbox()
iface = gr.Interface(fn = transcript_audio,inputs = audio_input,outputs = output_text,title ="Audio transcription app",description = "Upload the audio file")
iface.launch(server_name="0.0.0.0", server_port= 5005)

* Running on local URL:  http://0.0.0.0:5005
* To create a public link, set `share=True` in `launch()`.


Device set to use cpu
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


In [14]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from groq import Groq
api_key = "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"
client = Groq(api_key = api_key)
llm = ChatGroq(
    model = "llama-3.3-70b-versatile",
    groq_api_key = api_key,
    temperature = 0.7,
    max_tokens = 500
)
def remove_non_ascii(text):
    return " ".join(i for i in text if ord(i) < 128)
def product_assistant(ascii_transcript):
    """Process financial terminology using Groq"""
    system_prompt = """You are an intelligent assistant specializing in financial products;
    your task is to process transcripts of earnings calls, ensuring that all references to
    financial products and common financial terms are in the correct format. For each
    financial product or common term that is typically abbreviated as an acronym, the full term 
    should be spelled out followed by the acronym in parentheses. For example, '401k' should be
    transformed to '401(k) retirement savings plan', 'HSA' should be transformed to 'Health Savings Account (HSA)', 'ROA' should be transformed to 'Return on Assets (ROA)', 'VaR' should be transformed to 'Value at Risk (VaR)', and 'PB' should be transformed to 'Price to Book (PB) ratio'. Similarly, transform spoken numbers representing financial products into their numeric representations, followed by the full name of the product in parentheses. For instance, 'five two nine' to '529 (Education Savings Plan)' and 'four zero one k' to '401(k) (Retirement Savings Plan)'. However, be aware that some acronyms can have different meanings based on the context (e.g., 'LTV' can stand for 'Loan to Value' or 'Lifetime Value'). You will need to discern from the context which term is being referred to and apply the appropriate transformation. In cases where numerical figures or metrics are spelled out but do not represent specific financial products (like 'twenty three percent'), these should be left as is. Your role is to analyze and adjust financial product terminology in the text. Once you've done that, produce the adjusted transcript and a list of the words you've changed"""
    response = client.chat.completions.create(
        model = "llama-3.3-70b-versatile",
        messages=[
            {"role":"system","content":system_prompt},
            {"role" :"user","content":ascii_transcript}
        ],
        temperature= 0.5,
        max_tokens= 2000
    )
    return response.choices[0].message.content
template = """
Generate meeting minutes and a list of tasks based on the provided context.

Context:
{context}

Meeting Minutes:
- Key points discussed
- Decisions made

Task List:
- Actionable items with assignees and deadlines
"""
prompt = ChatPromptTemplate.from_template(template)
chain = (
    {"context":RunnablePassthrough()}
    |prompt
    |llm
    |StrOutputParser()
)
def transcript_audio(audio_file):
    """Main function to process audio and generate meeting minutes"""
    
    if audio_file is None:
        return "Please upload an audio file", None
    
    try:
        # Step 1: Transcribe audio using Whisper
        print("🎙️ Transcribing audio...")
        pipe = pipeline(
            "automatic-speech-recognition",
            model="openai/whisper-medium",
            chunk_length_s=30,
        )
        raw_transcript = pipe(audio_file, batch_size=8)["text"]
        
        # Step 2: Clean transcript
        print("🧹 Cleaning transcript...")
        ascii_transcript = remove_non_ascii(raw_transcript)

        # Step 3: Adjust financial terminology
        print("💼 Adjusting financial terminology...")
        adjusted_transcript = product_assistant(ascii_transcript)
        
        # Step 4: Generate meeting minutes and tasks
        print("📝 Generating meeting minutes...")
        result = chain.invoke(adjusted_transcript)

        # Step 5: Write the result to a file for downloading
        output_file = "meeting_minutes_and_tasks.txt"
        with open(output_file, "w") as file:
            file.write(result)

        print("✅ Complete!")
        # Return the textual result and the file for download
        return result, output_file
        
    except Exception as e:
        return f"Error: {str(e)}", None


#######------------- Gradio Interface-------------#######

audio_input = gr.Audio(sources="upload", type="filepath", label="Upload your audio file")
output_text = gr.Textbox(label="Meeting Minutes and Tasks", lines=20)
download_file = gr.File(label="Download the Generated Meeting Minutes and Tasks")

iface = gr.Interface(
    fn=transcript_audio,
    inputs=audio_input,
    outputs=[output_text, download_file],
    title="🤖 AI Meeting Assistant (Powered by Groq)",
    description="""
    Upload an audio file of a meeting. This tool will:
    1. 🎙️ Transcribe the audio using Whisper
    2. 💼 Fix financial product terminology
    3. 📝 Generate structured meeting minutes
    4. ✅ Extract actionable tasks with assignees
    
    **Powered by Groq's Llama 3.3 70B**
    """,
    examples=[
        # Add example audio files if you have them
    ]
)

# Launch the interface
iface.launch(server_name = "0.0.0.0", server_port = 5015)  

* Running on local URL:  http://0.0.0.0:5015
* To create a public link, set `share=True` in `launch()`.


🎙️ Transcribing audio...


Device set to use cpu
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


In [15]:
exit